# DPP Simulation Notebook

This notebook simulates a Determinantal Point Process (DPP) by importing reusable
functions from the `dpp` package (`dpp/__init__.py`).

## Simulation assumptions
1. Each candidate item has a feature vector $x_i \in \mathbb{R}^d$.
2. Item quality is a positive scalar $q_i > 0$.
3. Pairwise similarity is modeled by an RBF kernel:
   $$S_{ij} = \exp\left(-\frac{\|x_i-x_j\|^2}{2\sigma^2}\right).$$
4. The L-ensemble kernel is:
   $$L = \mathrm{diag}(q) S \mathrm{diag}(q), \quad L_{ij}=q_iS_{ij}q_j.$$
5. MAP inference is tracked by Schur-complement and rank-1 inverse updates.

## Function guide
- `generate_item_features`: creates synthetic candidate vectors.
- `quality_scores`: converts vectors into positive quality terms.
- `rbf_similarity`: computes pairwise similarity matrix $S$.
- `build_l_kernel`: builds DPP kernel matrix $L$.
- `map_inference_rank1_updates`: runs greedy MAP and stores each rank-1 update state.
- `save_map_inference_trace_gif`: saves one GIF with selected items and matrix evolution.

In [1]:
from pathlib import Path

import numpy as np

from dpp import (
    build_l_kernel,
    generate_item_features,
    map_inference_rank1_updates,
    quality_scores,
    rbf_similarity,
    save_map_inference_trace_gif,
)

In [2]:
# Configuration
N_ITEMS = 25
FEATURE_DIM = 2
SIGMA = 1.2
N_TRIALS = 20
SEED = 42

# Build DPP components
features = generate_item_features(N_ITEMS, FEATURE_DIM, seed=SEED)
qualities = quality_scores(features)
similarity = rbf_similarity(features, sigma=SIGMA)
l_kernel = build_l_kernel(qualities, similarity)

print('feature shape:', features.shape)
print('L-kernel shape:', l_kernel.shape)
print('L-kernel rank (numerical):', np.linalg.matrix_rank(l_kernel))

feature shape: (25, 2)
L-kernel shape: (25, 25)
L-kernel rank (numerical): 25


In [3]:
# Inspect diagonal entries as initial MAP gains when Y is empty.
initial_gains = np.diag(l_kernel)

print('Top-5 initial gains diag(L):')
top_idx = np.argsort(initial_gains)[::-1][:5]
for idx in top_idx:
    print(f'  item={idx}, gain={initial_gains[idx]:.3f}')

Top-5 initial gains diag(L):
  item=2, gain=5.141
  item=15, gain=4.214
  item=17, gain=1.442
  item=11, gain=1.352
  item=1, gain=1.305


In [4]:
# Run MAP inference with rank-1 inverse updates and keep step-by-step history.
map_subset, map_history = map_inference_rank1_updates(
    l_kernel=l_kernel,
    max_length=8,
    eps=1e-10,
)

print('MAP subset:', map_subset)
print('Recorded MAP steps:', len(map_history))

# Save a single GIF that includes selected items + matrix evolution.
output_gif = Path('outputs') / 'dpp_map_rank1_trace.gif'
saved_path = save_map_inference_trace_gif(
    features=features,
    qualities=qualities,
    map_history=map_history,
    output_path=output_gif,
    fps=1,
)
print('Saved unified GIF path:', saved_path)

MAP subset: [2, 15, 17, 0, 19, 11, 16, 6]
Recorded MAP steps: 8
Saved unified GIF path: /Users/wararaki/workspace/scrapbox8/math_sample/simulate-dpp-algorithm/outputs/dpp_map_rank1_trace.gif
